###Ingest drivers.json file
1. Read the file using spark dataframe reader API
2. Define and enforce schema (preserve the nested structure)
3. Add Metadata Columns
    * Source file
    * Ingestion Timestamp
4. Write the bronze delta table


In [0]:
%run ../00-common/01.environment-config


In [0]:
%run ../00-common/02.bronze-helpers

In [0]:
source_file = f"{landing_folder_path}/drivers.json"
table_name = f"{catlog_name}.{bronze_schema}.drivers"

###Step 1 - Read the JSON file using the dataframe reader API

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, DateType

name_schema = StructType([
    StructField('givenName', StringType()),
    StructField('familyName', StringType()),
])

driver_schema = StructType([
    StructField('driverId', StringType()),
    StructField('name', name_schema),
    StructField('dateOfBirth', DateType()),
    StructField('nationality', StringType()),
    StructField('url', StringType()),
])

In [0]:
driver_df = (
    spark.read
    .format('json')
    .schema(driver_schema)
    .option('mode', 'FAILFAST')
    .load(source_file)
)

In [0]:
display(driver_df)

In [0]:
driver_df_final = add_ingestion_metadata(driver_df)

In [0]:
display(driver_df_final)

In [0]:
(
    driver_df_final
    .write
    .format('delta')
    .mode('overwrite')
    .saveAsTable(table_name)
)

In [0]:
display(spark.table(table_name))